# 04. Residual Anomaly Detection & Persistence Filtering

This notebook detects power generation anomalies using **Residual Deviation Thresholding** and applies a **Rolling Window Persistence Filter** (40-minute window) to suppress transient cloud shading noise.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.append(str(Path("..").resolve()))

from src.config import CLEANED_DATA_PATH
from src.predict import predict_expected_power
from src.anomaly import detect_anomalies

sns.set_theme(style="whitegrid")

## 1. Compute Expected Power & Detect Residual Anomalies

In [ ]:
df_clean = pd.read_csv(CLEANED_DATA_PATH)
df_pred = predict_expected_power(df_clean)
df_anomaly, threshold_val = detect_anomalies(df_pred)

print(f"95th Percentile Relative Deviation Threshold: {threshold_val:.2f}%")
print(f"Raw Transient Anomalies Detected: {df_anomaly['raw_anomaly'].sum()}")
print(f"Persistent Fault Anomalies (>=40 mins): {df_anomaly['persistent_anomaly'].sum()}")

## 2. Visualize Anomalies on Power Output Curve

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df_anomaly['timestamp'][:400], df_anomaly['total_ac_power_kw'][:400], label='Actual AC Power (kW)', color='#2c3e50', alpha=0.7)
plt.plot(df_anomaly['timestamp'][:400], df_anomaly['expected_power_kw'][:400], label='Expected AC Power (kW)', color='#3498db', linestyle='--')

# Highlight Anomalies
anom_points = df_anomaly[:400][df_anomaly[:400]['persistent_anomaly']]
plt.scatter(anom_points['timestamp'], anom_points['total_ac_power_kw'], color='#e74c3c', s=50, label='Persistent Anomaly (Fault)', zorder=5)

plt.xlabel('Timestamp')
plt.ylabel('Power (kW)')
plt.title('NISE Solar Telemetry: Residual Anomaly Detection with Persistence Filter')
plt.legend()
plt.tight_layout()
plt.show()